In [1]:
#Libraries
!pip install -q transformers datasets evaluate seqeval accelerate

import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00
Using device: cuda


In [5]:
# Task 1: Dataset Selection
# We use CoNLL-2003 which contains both POS and Chunk tags.
dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")

# Sample data to make training fast on Colab
train_dataset = dataset["train"].select(range(2500))
val_dataset = dataset["validation"].select(range(500))

# Extract POS features and labels
pos_feature = dataset["train"].features["pos_tags"]
label_names = pos_feature.feature.names

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

print(f"Dataset Loaded! Number of classes: {len(label_names)}")
print(f"Label Categories: {label_names[:10]}... (Total: {len(label_names)})")

conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset Loaded! Number of classes: 47
Label Categories: ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``']... (Total: 47)


In [6]:
# Task 2: Data Preprocessing & Label Alignment
model_name = "distilbert-base-cased" # Casing matters for POS tags!
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    # Tokenize the words, keeping track of subwords
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set their label to -100 so they are ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # Set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For subwords, also set the label to -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Aligning labels with subwords...")
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_val = val_dataset.map(tokenize_and_align_labels, batched=True)
print("Label alignment complete!")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Aligning labels with subwords...


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Label alignment complete!


In [7]:
# Task 3 & 5: Model Setup and Seqeval Metric Setup
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100)
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Load the Model
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
# Task 4: Training with Hugging Face Trainer
# Data collator pads inputs and labels properly (-100 for labels)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./pos_tagging_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,  # <-- FIXED HERE: Changed 'tokenizer' to 'processing_class'
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

print("\n--- Final Evaluation Metrics ---")
print(trainer.evaluate())

Starting training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.393080,0.877611,0.860610,0.869028,0.911228
2,No log,0.314573,0.901470,0.873970,0.887507,0.924035
3,No log,0.286287,0.904178,0.886662,0.895334,0.928947


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni


--- Final Evaluation Metrics ---


{'eval_loss': 0.2862870991230011, 'eval_precision': 0.9041780199818347, 'eval_recall': 0.88666221331552, 'eval_f1': 0.8953344575604273, 'eval_accuracy': 0.9289473684210526, 'eval_runtime': 0.9693, 'eval_samples_per_second': 515.824, 'eval_steps_per_second': 64.994, 'epoch': 3.0}


In [10]:
# Task 6: Inference on custom sentences
from transformers import pipeline

# We use the trained model to create a token classification pipeline
pos_pipeline = pipeline("token-classification", model=model.to("cpu"), tokenizer=tokenizer, aggregation_strategy="simple")

text = "Raj works at Innomatics in Hyderabad and loves NLP."
print(f"Input text: {text}\n")

predictions = pos_pipeline(text)

print("Predicted POS Tags:")
for pred in predictions:
    print(f"Word: {pred['word']:<12} | Tag: {pred['entity_group']}")

Input text: Raj works at Innomatics in Hyderabad and loves NLP.

Predicted POS Tags:
Word: Raj          | Tag: NNP
Word: works        | Tag: VBZ
Word: at           | Tag: IN
Word: Inn          | Tag: NNP
Word: ##oma        | Tag: NN
Word: ##tics       | Tag: NNS
Word: in           | Tag: IN
Word: Hyderabad    | Tag: NNP
Word: and          | Tag: CC
Word: loves        | Tag: VBZ
Word: NLP          | Tag: NNP
Word: .            | Tag: .


### Task 7 & 8: Comparison and Report

#### Differences between POS Tagging and Chunking
1. **POS Tagging (Part-of-Speech):** This is a grammar-level task where every single word is assigned a grammatical label (Noun, Verb, Adjective). It is relatively "Easy" because it usually relies mostly on the word itself and its immediate neighbors. Example: "Innomatics" -> Proper Noun (NNP).
2. **Chunking (Phrase Detection):** This is a phrase-level task (Medium difficulty). Instead of tagging individual words, chunking groups consecutive words into logical phrases (e.g., Noun Phrases, Verb Phrases). Example: "The quick brown fox" -> Noun Phrase (NP).

#### Challenges Faced & Insights
* **Subword Tokenization (-100 Label):** The biggest challenge in Token Classification is that Transformers split unknown words into subwords (e.g., "Innomatics" might be split into "Inn", "##omatics"). We cannot give the subword "##omatics" its own POS tag without confusing the loss function. The solution was carefully aligning the `word_ids` and masking subwords with `-100` so PyTorch ignores them during gradient descent.
* **Casing Matters:** Unlike standard sentiment analysis where we lowercased everything, using a `cased` model was crucial here. Capital letters are massive indicators for Proper Nouns (NNP) in POS tagging (like "Raj", "Innomatics", and "Hyderabad").
* **Performance:** DistilBERT learns POS tags incredibly fast because grammatical structure is deeply embedded in the pre-trained weights of the model.